# Error Handling and Validation



```{contents}
:local:
:depth: 2
```


```{index} Decide What a Failure Means Before Catching It
```



```{index} Follow One Failed Calculation Through Its Caller
```

## Follow One Failed Calculation Through Its Caller


A business calculation should either return a meaningful value or make failure explicit. A returned zero could mean that no payment is due; it must not secretly mean that the calculation never succeeded. Use the method-contract ideas from Chapter 5 to decide which inputs the calculation accepts, and the control-flow ideas from Chapters 3 and 4 to follow what happens next.

The general pattern below is schematic. Replace the operation, exception type, and response with the policy for the actual task. The braces delimit blocks. `catch (ExceptionType name)` selects a compatible exception and gives that exception a local name. It does not execute after every successful operation.

```text
try
{
    operationThatMayThrow();
    workThatRequiresSuccess();
}
catch (SpecificExceptionType problem)
{
    permittedFailureResponse();
}
finally
{
    workRequiredWhenLeavingThisTry();
}
```

There can be several catch blocks, ordered from more specific types to more general ones. Only the first matching catch runs for that thrown exception. A `finally` block is optional, as are catch blocks when a try has a finally. Cleanup is not a substitute for handling the failure, and catching a failure does not restart the interrupted statement.

Consider distributing an invoice amount across a positive number of periods. The method rejects zero periods before division. Its caller knows how to explain that particular argument failure. No file API or collection is needed to understand this example.


In [1]:
using System;
static decimal PerPeriod(decimal total, int periods)
{
    if (periods <= 0)
        throw new ArgumentOutOfRangeException(nameof(periods), "Use a positive period count.");
    return total / periods;
}
static void ShowPlan(int periods)
{
    try
    {
        Console.WriteLine($"Begin {periods}");
        decimal amount = PerPeriod(120m, periods);
        Console.WriteLine($"Amount {amount:0.00}");
    }
    catch (ArgumentOutOfRangeException)
    {
        Console.WriteLine("Choose at least one period.");
    }
    finally
    {
        Console.WriteLine("End attempt");
    }
}
ShowPlan(3);
ShowPlan(0);


Begin 3
Amount 40.00
End attempt
Begin 0
Choose at least one period.
End attempt


Read the two attempts separately. For three periods, the call returns 40 and the success message runs. For zero periods, `throw` ends the method before division. Control leaves the unfinished try block, enters the matching catch, and then executes finally. There is no `Amount` line for the failed attempt.

| Attempt | Does the method return an amount? | Runs the catch? | Runs finally? |
| --- | --- | --- | --- |
| Three periods | Yes, 40 | No | Yes |
| Zero periods | No | Yes | Yes |
| Negative periods | No | Yes | Yes |

The exception is created by `new ArgumentOutOfRangeException(...)`; `throw` transfers control. `nameof(periods)` records the parameter name without duplicating its spelling in a string. Do not catch an unrelated type merely because its name sounds like an error. An unmatched exception continues to an enclosing handler after this finally runs. The statement after the failed call is not resumed.

Printing from finally is used only to make control visible here. In Chapter 9, resource disposal will be a concrete cleanup responsibility. Finally does not mean that a result was produced, and it is not guaranteed to run after process termination or machine failure.


```{index} Guide a Repair Before Broadening the Failure Policy
```

## Guide a Repair Before Broadening the Failure Policy


First predict the complete output for `ShowPlan(1)` and `ShowPlan(-2)`. Identify the first statement that is not executed in the negative case. Then change the contract: a negative total is invalid, while zero total remains valid. Add an argument guard to the calculation, rather than returning a fake amount.

```{dropdown} Check the reasoning
One period prints Amount 120.00. Minus two periods prints the correction message and no amount. To reject a negative total, check `total < 0` and throw an appropriate argument exception before returning. Decide which specific handler the caller needs for that new failure; the existing out-of-range handler only covers that type. Returning zero for an invalid total would merge a failed request with a legitimate zero invoice.
```

Try valid zero, valid positive, and invalid negative totals with one and zero periods. Derive each expected outcome before running it. Later file-handling examples are previews of Chapter 9, where their APIs and ownership rules are taught; they are not prerequisites for this control-flow lesson.


```{index} Decide What a Failure Means Before Catching It
```

## Decide What a Failure Means Before Catching It



An employee enters a delivery quantity. There are at least three different problems the application might encounter: the text is not a number, the number violates a business rule, or a separate resource needed to complete the operation is unavailable. These situations can all prevent success, but they do not call for the same response.

Validation asks whether input satisfies a stated contract. Parsing asks whether text can be interpreted as a value. Exception handling responds to an exceptional transfer of control from an operation. A catch block is not a substitute for specifying which quantities are legal, and a successful parse is not a substitute for checking the domain.

| Input or event | Classification | Appropriate next decision |
| --- | --- | --- |
| Text `twelve` for an integer quantity | Representation failure | Request a whole-number input |
| Integer -2 for packages | Domain failure | Explain that quantity must be nonnegative |
| End of the input stream | No further input | Stop or cancel explicitly |
| A resource operation throws a known recoverable exception | Operation failure | Report context or choose an allowed fallback |

Work through the decisions in that order. First handle a missing input value. Then try to interpret available text. Only after a value exists can the program compare it with the business bounds. If you check a default zero after a failed parse without checking the Boolean success result, you can accidentally treat failed input as a legitimate zero-quantity order.

For related explanation, see the further reading below.[^development-reading]


```{index} Follow Control Through an Exception
```



```{index} Follow Control Through an Exception
```

## Follow Control Through an Exception



A try block marks the work whose failure policy you are handling. When an operation throws, normal execution of that block stops at that point; statements after the throwing operation are not completed. A matching catch handles the exception. A finally block provides cleanup when control leaves the try/catch in ordinary managed execution, including when a handled exception occurs. It does not promise that a killed process or failed machine will run your cleanup code.

Catch only a failure for which you have a meaningful policy. An empty catch that continues to print Success converts a visible failure into misleading behavior. Catching every exception and replacing it with zero may also destroy the distinction between an unavailable answer and a real zero result. File-related examples below preview Chapter 9; their resource APIs will be developed there, rather than assumed as prerequisites for validation.

Guided prediction: quantity zero is permitted, a negative quantity is rejected, and nonnumeric text asks for correction. What should the program report for `0`, `-1`, and `x`? Explain why all three should not share one generic Invalid message.

```{dropdown} Compare the outcomes
Zero is a valid value under this contract. Minus one is interpretable but outside the business domain. The letter x has no integer interpretation. Different messages identify what the user can change, while avoiding a false success result.
```


In [1]:
using System;
string text = "-2";
if (!int.TryParse(text, out int quantity)) Console.WriteLine("Enter a whole number.");
else if (quantity < 0) Console.WriteLine("Quantity must be nonnegative.");
else Console.WriteLine($"Accepted quantity: {quantity}");


Quantity must be nonnegative.


Programs fail in predictable ways: bad input, missing files, unexpected values, and invalid assumptions.
Good software does not pretend errors never happen — it handles them deliberately.

```{index} Validation vs. Exceptions
```



```{index} Validation vs. Exceptions
```

## Validation vs. Exceptions




- **Validation** checks expected bad input *before* doing risky work.
- **Exceptions** handle unexpected failures *at runtime*.

Validation should be your first line of defense. Exceptions are a fallback for when something still goes wrong.

| | Validation | Exception |
| --- | --- | --- |
| When | Before the risky operation | When the operation has already failed |
| Typical trigger | Empty text, negative quantity, missing field | Missing file, lost connection, bug in a called method |
| Cost | A comparison | Building an exception object and unwinding the stack |
| Example | `if (int.TryParse(text, out int n) && n >= 0)` | `catch (FileNotFoundException)` |

The two are not rivals: validate what you can predict, and leave `catch` for what you cannot.


```{index} Basic try / catch / finally
```



```{index} Basic try / catch / finally
```

## Basic `try` / `catch` / `finally`



The statement has three parts, and only the first is required to have company:

```csharp
try
{
    // statements that might throw
}
catch (SpecificException ex)      // runs only if that exception type was thrown
{
    // respond to that failure; ex.Message describes it
}
catch (AnotherException)          // more catch blocks, most specific first
{
}
finally                           // runs whether or not anything was thrown
{
    // cleanup: close resources, reset state
}
```

When a statement inside `try` throws, the rest of the `try` block is skipped, the first matching `catch` runs, and then `finally` runs. When nothing throws, the `catch` blocks are skipped and `finally` still runs. The example parses a number; in the browser there is no keyboard, so the cell falls back to the text `12x`, which is not a whole number:


In [ ]:
using System;

string raw = Console.ReadLine() ?? "12x";   // no keyboard in the browser: use a bad sample
try
{
    Console.WriteLine($"Parsing '{raw}'");
    int value = int.Parse(raw);
    Console.WriteLine($"You entered {value}");
}
catch (FormatException)
{
    Console.WriteLine("Input must be a whole number.");
}
catch (OverflowException)
{
    Console.WriteLine("Number is outside the valid int range.");
}
finally
{
    Console.WriteLine("Input attempt complete.");
}


The output is `Parsing '12x'`, `Input must be a whole number.`, and `Input attempt complete.` The line `You entered ...` never printed, because `int.Parse` threw before reaching it. Change the sample to `"2147483648"` and the second `catch` runs instead, since that value is one more than `int` can hold; change it to `"42"` and no `catch` runs at all, but `finally` still prints. Two mistakes: putting a `catch (Exception)` block *before* the specific ones, which the compiler rejects because the later blocks could never run, and putting cleanup only at the end of the `try` block, where a thrown exception skips it.


```{index} Prefer TryParse for User Input
```



```{index} Prefer TryParse for User Input
```

## Prefer `TryParse` for User Input



If invalid input is *expected*, avoid throwing exceptions entirely. `TryParse` reports success as a `bool` and delivers the parsed value through an `out` parameter, the mechanism Chapter 5 introduced:

```csharp
if (int.TryParse(text, out int value))   // true and value set when text is a whole number
{
    // use value
}
```

The `out` variable is declared inside the condition and is available afterward. In the browser the cell falls back to the sample `41`:


In [ ]:
using System;

string raw = Console.ReadLine() ?? "41";    // no keyboard in the browser: use a sample

if (int.TryParse(raw, out int age) && age >= 0)
{
    Console.WriteLine($"Age recorded: {age}");
}
else
{
    Console.WriteLine("Please enter a non-negative whole number.");
}


The output is `Age recorded: 41`. With the sample `"forty"` the first condition is false and the message prints; with `"-3"` parsing succeeds but the second condition fails, so the same message prints. No exception is created in either case, which is cheaper and reads as ordinary decision logic. The mistake to avoid is using `age` after `TryParse` returned false: it is set to `0` then, and `0` is a plausible age.


**Guideline:**
- Use `TryParse` for normal input flow where bad input is common.
- Use `try/catch` around operations that can fail unpredictably (file I/O, network, external dependencies).

```{index} File I/O with Safe Handling
```



```{index} File I/O with Safe Handling
```

## File I/O with Safe Handling



File operations fail for reasons outside the program's control, so they are the classic place for `try`. The `using` declaration disposes the reader when the block ends, whether normally or by an exception, which is why no `finally` is needed for the file itself. Catch blocks are listed most specific first: `FileNotFoundException` and `UnauthorizedAccessException` are both kinds of `IOException`, so the general one comes last. In the browser `scores.txt` does not exist, which is the case the first `catch` handles:


In [ ]:
string path = "scores.txt";

try
{
    using StreamReader reader = new StreamReader(path);
    string? line;
    while ((line = reader.ReadLine()) != null)
        Console.WriteLine(line);
}
catch (FileNotFoundException)
{
    Console.WriteLine($"Could not find file: {path}");
}
catch (UnauthorizedAccessException)
{
    Console.WriteLine("Permission denied while reading the file.");
}
catch (IOException ex)
{
    Console.WriteLine($"I/O error: {ex.Message}");
}

The output is `Could not find file: scores.txt`. Reverse the order and put `catch (IOException ex)` first: the compiler refuses with an error, because the two later blocks could never run. Create the file first with `File.WriteAllText(path, "90\n85")` at the top of the cell and the loop prints the two lines instead; the `catch` blocks are then skipped entirely.


```{index} Throwing Exceptions Intentionally
```



```{index} Throwing Exceptions Intentionally
```

## Throwing Exceptions Intentionally



Throwing is how a method refuses an argument it cannot honor. The statement creates an exception object with a message and hands control to the nearest matching `catch` up the call chain:

```csharp
throw new ArgumentException("why the argument is unacceptable", nameof(parameter));
```

`nameof(parameter)` records which parameter was wrong without retyping its name as a string. The second call below triggers the throw, and the caller catches it so the cell can finish:


In [ ]:
using System;

Console.WriteLine(Divide(10, 2));
try
{
    Console.WriteLine(Divide(10, 0));   // throws before anything is printed
}
catch (ArgumentException ex)
{
    Console.WriteLine($"Caught: {ex.Message}");
}

static double Divide(double numerator, double denominator)
{
    if (denominator == 0)
        throw new ArgumentException("Denominator cannot be zero.", nameof(denominator));

    return numerator / denominator;
}


The output is `5` and then `Caught: Denominator cannot be zero. (Parameter 'denominator')`; the runtime appended the parameter name to the message. Returning a sentinel such as `-1` or `double.NaN` instead would let a bad call continue silently and surface as a wrong total somewhere far away, which is harder to trace than an exception thrown at the point of misuse. Throw for arguments the method's contract rules out; use `TryParse`-style results for input that is merely often wrong.


```{index} Error Message Quality
```



```{index} Error Message Quality
```

## Error Message Quality




| Bad | Better |
|-----|--------|
| `"Error happened"` | `"Could not open 'scores.txt'. Check that the file exists and that you have read permission."` |

Good error messages are: **specific**, **user-actionable**, and free of raw internal noise.

```{index} Common Mistakes
```



```{index} Common Mistakes
```

## Common Mistakes



- **Catching `Exception` everywhere.** `catch (Exception)` also catches bugs such as `NullReferenceException` and hides them behind a generic message. Catch the types you can actually respond to.
- **Swallowing exceptions.** An empty `catch { }` makes the failure disappear and the program continue with bad state. At minimum, report what happened.
- **Using exceptions for normal control flow.** Parsing user input with `try { int.Parse(...) }` throws on every typo; `TryParse` is the right tool when bad input is expected.
- **Showing stack traces to end users.** The trace helps the developer, not the customer. Log it, and show the user a specific, actionable message as in the table above.

```csharp
try { total = int.Parse(text); }
catch { }                        // mistake: the failure vanishes and total keeps its old value
```


```{index} Practice
```

## Practice




1. Read a value from the console and keep prompting until a valid `double` is entered.
2. Open a file path provided by the user; print a friendly message for: file not found, access denied, and unknown I/O error.
3. Add argument checks to one of your existing methods and throw `ArgumentException` when input is invalid.

## Rethrow Without Losing the Failure Origin



```{index} gap revision: rethrow without losing the failure origin
```


A layer may record a failure but still be unable to recover. The caller must then receive the failure, with evidence of where it began. Inside a catch block, `throw;` rethrows the current exception and preserves its original stack trace. `throw ex;` throws the caught object again from the new throw statement, resetting the visible origin there. Logging a message is not recovery.

General patterns (schematic):

```csharp
catch (Exception ex)
{
    Console.WriteLine(ex.Message);
    throw; // propagate the current failure, retaining the original trace
}
```

Use a specific catch type when you can identify the failure. The broad catch above only illustrates propagation, not a recommendation to swallow every failure. Throwing a new exception with the original as its inner exception is another contract, useful when adding meaningful context; do not silently replace the evidence.

| Action in catch | Caller receives a failure? | Diagnostic consequence |
| --- | --- | --- |
| `throw;` | Yes. | Retains original throw information. |
| `throw ex;` | Yes. | Resets stack origin at this throw. |
| Only print a message | No, unless another failure occurs. | Caller may wrongly assume success. |

A `finally` block should complete cleanup. If cleanup throws while another exception is propagating, the new failure can mask the original. Keep cleanup reliable and avoid treating `finally` as a place to return a replacement result.

See the language/API reference for this contract.[^gap-rethrow]


In [1]:
using System;
static void Validate(int quantity)
{
    if (quantity <= 0) throw new ArgumentOutOfRangeException(nameof(quantity));
}
static void Submit(int quantity)
{
    try { Validate(quantity); }
    catch (ArgumentOutOfRangeException) { Console.WriteLine("Submission failed"); throw; }
}
try { Submit(0); }
catch (ArgumentOutOfRangeException) { Console.WriteLine("Caller received failure"); }


Submission failed
Caller received failure


**Guided practice.** Explain why the caller still executes its catch. In VS Code, inspect the exception stack and locate `Validate`. Predict what evidence changes if `throw;` becomes `throw ex;`. Keep that variation labeled as a diagnostic mistake, not a recommended repair.


```{rubric} Footnotes
```

[^gap-rethrow]: [Microsoft: Rethrow to preserve stack details](https://learn.microsoft.com/dotnet/fundamentals/code-analysis/quality-rules/ca2200).


[^development-reading]: Further reading: [Microsoft, Exceptions and Exception Handling](https://learn.microsoft.com/en-us/dotnet/csharp/fundamentals/exceptions/). The worked business scenarios and guided reasoning here are original teaching examples.
